# Problem 4: Gene Finding

---

## Background

In the previous problems, you built two separate tools: an ORF finder that locates open reading frames in a DNA sequence, and a CpG island detector that identifies regions with high CpG dinucleotide frequency. Individually, each tool produces many results — most of which are not real genes. Your ORF finder flags every sequence between a start and stop codon, including many short, spurious matches. Your CpG island detector finds GC-rich regions, but not all of them are near genes.

In this problem, you will combine these tools into a simple **gene finder** — a program that uses multiple lines of evidence to predict which ORFs are most likely to be real, expressed genes. This is a simplified version of the approach used by real gene prediction software, where combining multiple signals dramatically reduces false positives.

### Why These Criteria Work Together

Real protein-coding genes tend to share certain features:

- They begin with a **start codon** (`ATG`), which signals the ribosome to begin translation.
- They encode proteins of meaningful length. Very short ORFs (under ~30 amino acids) are overwhelmingly likely to occur by chance and are rarely functional genes.
- They are often associated with **CpG islands**, which tend to occur in the promoter regions upstream of and overlapping with the 5' end of genes. If a CpG island overlaps the beginning of an ORF, that is evidence the ORF may have a real promoter driving its expression.

No single criterion is sufficient on its own, but together they provide a much stronger signal.

## Task

Write a gene finder function in `src/module_2_assignment/gene_finder.py` that imports and uses your existing ORF finder and CpG island detector. The function should identify candidate genes by applying the following criteria.

### Gene Finding Criteria

An ORF is classified as a **candidate gene** if it meets **all four** of the following conditions:

1. **Starts with ATG.** The ORF must begin with a start codon (`ATG`).

2. **Minimum length of 96 bp (32 amino acids).** The ORF must be at least 96 base pairs long, including the start and stop codons. In random DNA, you would expect an ATG followed by an in-frame stop codon roughly every 60–90 bp, so requiring 96 bp filters out most of this noise.

3. **Overlaps a CpG island.** At least one CpG island detected by your CpG island finder must overlap with the ORF. Any amount of overlap counts — the CpG island does not need to be entirely within the ORF or vice versa.

4. **CpG island overlaps the first third of the ORF.** The overlapping CpG island must specifically overlap with the **first third** of the ORF. This reflects the biological expectation that CpG islands are associated with promoter regions near the 5' (start) end of genes. To compute the first third: if an ORF spans positions *start* to *end*, its first third covers positions *start* to *start + (end − start) / 3*.

### Searching Both Strands

Genes can be encoded on either strand of DNA. Your gene finder should search all **six reading frames** — three on the forward strand and three on the reverse complement. To search the reverse strand, compute the reverse complement of the input sequence and run the same analysis on it.

## Example

The following illustrative example shows how the four criteria work together to identify a candidate gene.

![Gene Finding Example](assets/gene-finding.png)

**ORF A** (positions 60–420, 360 bp): Starts with ATG, exceeds 96 bp, and a CpG island (80–320) overlaps its first third (60–160). All four criteria are met → **candidate gene**.

**ORF B** (positions 380–540, 160 bp): Starts with ATG and exceeds 96 bp, but no CpG island overlaps it (the nearest CpG island ends at 320, 60 bp away) → **rejected**.

**ORF C** (positions 10–500, 490 bp): Starts with ATG, exceeds 96 bp, and a CpG island (280–420) overlaps the ORF — but only in its middle, not its first third (10–173). The CpG island is too far from the promoter region → **rejected**.

## Implementation

Your function should:

1. Take a DNA sequence string as input.
2. Run your ORF finder to identify all ORFs that start with ATG and are at least 96 bp long.
3. Run your CpG island detector on the same sequence.
4. Filter ORFs to keep only those where a CpG island overlaps the first third of the ORF.
5. Repeat steps 2–4 on the reverse complement of the sequence.
6. Return the list of candidate genes with their positions, strand, and length.

Report results in the following format:

```
Gene candidate: position X to Y (strand: +/-, length: Z bp)
```

Use **1-based indexing** relative to the original forward strand sequence.

> **Tip:** Test your gene finder on a small sequence first. You can construct a test sequence that contains a known ORF overlapping a CpG island to verify your logic before running on the full chromosome.

### Run on Chromosome 21

Run your gene finder on Human Chromosome 21. Use the same FASTA file you downloaded for Problem 3 (CpG islands) (`data/` directory).

> **Reminder:** The Chromosome 21 file should be in your `.gitignore` and not committed to GitHub. If you haven't already done so, add `data/NC_000021*` to your `.gitignore`.

The current estimate for the number of protein-coding genes on Chromosome 21 is approximately **200–300** [1, 2]. The original sequencing of Chromosome 21 identified 225 genes and 59 pseudogenes [3]. Your simplified gene finder will not match this number exactly — that's expected. Use the known estimate as a reference point to evaluate whether your results are in a reasonable range:

- **Far too many candidates** (thousands) → your criteria may be too permissive, or you may not be combining the filters correctly.
- **Far too few candidates** (under 50) → your criteria may be too strict, or your CpG island detector may be missing regions.
- **In the hundreds** → you're in the right ballpark for a simplified approach.

### Unit Testing

Write at least **one unit test** in `tests/test_gene_finder.py`. Consider testing:

- A short sequence with a known ORF that overlaps a CpG island → should be detected.
- A short ORF (under 96 bp) that overlaps a CpG island → should be rejected.
- An ORF that starts with ATG but has no overlapping CpG island → should be rejected.
- An ORF where the CpG island only overlaps the last third → should be rejected.

Run your tests from the command line:

```
pytest tests/test_gene_finder.py
```

## Analysis

After running your gene finder on Chromosome 21, reflect on your results. Consider the following:

- How many candidate genes did your gene finder identify? How does this compare to the known estimate of 200–300 protein-coding genes?
- Which of the four criteria had the biggest effect on filtering? (You might try running with fewer criteria to see how the count changes.)
- What additional criteria could improve accuracy? Consider what real gene finders use that your simplified version does not. Try asking an AI tool (such as Claude, Gemini, or ChatGPT) what other signals are used in gene prediction and discuss which ones you think would be most effective.

## References

1. Chromosome 21. MedlinePlus Genetics. National Library of Medicine. https://medlineplus.gov/genetics/chromosome/21/
2. Gardiner, K. et al. The proteins of human chromosome 21. *Current Genomics*, 7(3), 169–174 (2006).
3. Hattori, M. et al. The DNA sequence of human chromosome 21. *Nature*, 405, 311–319 (2000). https://doi.org/10.1038/35012518

In [1]:
from module_2_assignment.fasta_parser import read_fasta
from module_2_assignment.gene_finder import find_genes
import time
file_path= "../data/NC_000021.fasta"
counter=0
start_time= time.perf_counter()
for description, dna_sequence in read_fasta(file_path):
    print(f"Processing DNA sequence: {description}")
    genes_candidates= find_genes(dna_sequence, description)
    print(f"DNA sequence description: {description}")
    counter+= len(genes_candidates)
    for gene in genes_candidates:
        start= gene[0]
        stop= gene[1]
        length= gene[2]
        strand= "+" if gene[3]==1 else "-"
        print(f"Gene candidate: position {start} to {stop} (strand: {strand}, length: {length} bp)")
    print()
end_time= time.perf_counter()
print(f"Time taken to process the file: {end_time - start_time:.2f} seconds")
print(f"Total number of gene candidates found: {counter}")


Processing DNA sequence: NC_000021.9 Homo sapiens chromosome 21, GRCh38.p14 Primary Assembly
Starting gene finder...
Positive strand ORFs found: 719,565
Negative strand ORFs found: 749,235
CpG islands found: 5,812
DNA sequence description: NC_000021.9 Homo sapiens chromosome 21, GRCh38.p14 Primary Assembly
Gene candidate: position 5014726 to 5015205 (strand: +, length: 480 bp)
Gene candidate: position 5014846 to 5015205 (strand: +, length: 360 bp)
Gene candidate: position 5014885 to 5015205 (strand: +, length: 321 bp)
Gene candidate: position 5014945 to 5015205 (strand: +, length: 261 bp)
Gene candidate: position 5015011 to 5015205 (strand: +, length: 195 bp)
Gene candidate: position 5019604 to 5019936 (strand: +, length: 333 bp)
Gene candidate: position 5021086 to 5021220 (strand: +, length: 135 bp)
Gene candidate: position 5022901 to 5023053 (strand: +, length: 153 bp)
Gene candidate: position 5026225 to 5026734 (strand: +, length: 510 bp)
Gene candidate: position 5026306 to 5026734 

### Analysis

- How many candidate genes did your gene finder identify? How does this compare to the known estimate of 200–300 protein-coding genes?
 
With my current gene-finding logic, I identified approximately 21,000 candidate genes, which is much higher than the known estimate of approximately 200–300 protein-coding genes on chromosome 21. I feel this large difference is due to the fact that the gene finder uses relatively simple criteria. It considers ORFs in all six reading frames, including overlapping and nested ORFs, and does not use many of the biological signals that distinguish real genes from ORFs that occur by chance. Therefore, I think the algorithm produces many false-positive gene candidates.

- Which of the four criteria had the biggest effect on filtering? (You might try running with fewer criteria to see how the count changes.)

Based on my implementation, the CpG-island overlap criterion appeared to have the largest filtering effect. There are many ORFs of at least 96 bp throughout the chromosome, but requiring an ORF to overlap a CpG island removes many of them because CpG islands occur only in particular genomic regions. The additional requirement that the CpG island overlap the first third of the ORF further reduced the number of candidates.

- What additional criteria could improve accuracy? Consider what real gene finders use that your simplified version does not. Try asking an AI tool (such as Claude, Gemini, or ChatGPT) what other signals are used in gene prediction and discuss which ones you think would be most effective.

I think several additional criteria could improve the accuracy of the simplified gene finder. One simple improvement would be to increase the minimum ORF length beyond 96 bp, which would eliminate more short ORFs that occur randomly. The CpG requirement could also be made stricter, for example by requiring a stronger or larger CpG-rich region near the beginning of the candidate gene. ALso, maybe using additional biological signals such as promoter motifs, splice sites, or conservation across species could help identify real genes.

